# SkillNet AI Scientist: From Task to Discovery

This notebook demonstrates the **SkillNet** workflow where an AI Agent autonomously plans and executes a scientific mission.

## Workflow Lifecycle
1.  **Task Definition**: User provides a high-level research goal.
2.  **AI Planning**: The Agent decomposes the goal into logical steps.
3.  **Skill Discovery**: The Agent searches the **SkillNet** for specialized skills matching each step.
4.  **Acquisition & Orchestration**: Skills are downloaded, validated, and composed into a pipeline.
5.  **Execution**: The skills interact to generate scientific results.


In [ ]:
# Install SkillNet-AI package and dependencies
%pip install skillnet-ai
%pip install scanpy requests leidenalg matplotlib seaborn pandas


^C


In [ ]:
import shutil
from IPython.display import display, Markdown

# Import the actual SkillNet Client
from skillnet_ai import SkillNetClient

print("✅ SkillNet AI Package loaded successfully.")


✅ SkillNet AI Package loaded successfully.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import requests
import json
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure Scanpy settings for reproducibility
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Set OpenAI API Key for SkillNet Evalaution & Analysis
# Replace with your actual key if available
os.environ["API_KEY"] = "your-api-key-here" 
os.environ["BASE_URL"] = "https://api.openai.com/v1"


In [40]:
# 1. Initialize SkillNet Agent
client = SkillNetClient()
print("🤖 **AI Scientist Agent Initialized**")

# 2. Task Definition & AI Planning
USER_MISSION = "Analyze single-cell RNA-seq data to identify potential cancer therapeutic targets, validate them against clinical databases, and write a summary report."
print(f"\n📝 **User Mission**: \"{USER_MISSION}\"")

# (Simulated AI Planning Step) - Updated plan to match available/searchable skills
plan = [
    {
        "step": 1, 
        "phase": "Data Processing", 
        "query": "cellxgene", # Search query for data source
        "expected_skill": "cellxgene-census" 
    },
    {
        "step": 2, 
        "phase": "Mechanism Analysis", 
        "query": "kegg", 
        "expected_skill": "kegg-database"
    },
    {
        "step": 3, 
        "phase": "Target Validation", 
        "query": "gget", 
        "expected_skill": "gget"
    },
    {
        "step": 4, 
        "phase": "Reporting", 
        "query": "scientific writing", 
        "expected_skill": "citation-management"
    }
]

# Directory to store our skills
SKILLS_LIB = "./active_skills_library"
if os.path.exists(SKILLS_LIB): shutil.rmtree(SKILLS_LIB)
os.makedirs(SKILLS_LIB, exist_ok=True)

active_skills = {}

print("\n🚀 **SkillNet Orchestration Started**")

for task in plan:
    print(f"\n--- [Phase {task['step']}: {task['phase']}] ---")
    
    # A. SEARCH
    print(f"🔍 Searching SkillNet for: '{task['query']}'...")
    try:
        results = client.search(q=task['query'], limit=1)
    except Exception as e:
        results = []
        print(f"   (Search unavailable: {e})")

    if results:
        best_skill = results[0]
        print(f"   ✅ Found Skill: {best_skill.skill_name} (Stars: {best_skill.stars})")
        skill_url = best_skill.skill_url
        skill_name = best_skill.skill_name
    else:
        # Fallback if no network or local skill not indexed
        print(f"   ⚠️ Skill not found via search. Retrieving from local registry: '{task['expected_skill']}'")
        skill_name = task['expected_skill']
        skill_url = None 

    # B. DOWNLOAD
    target_path = os.path.join(SKILLS_LIB, skill_name)
    if skill_url:
        print(f"   ⬇️ Downloading to: {target_path}...")
        try:
            client.download(url=skill_url, target_dir=SKILLS_LIB)
        except Exception as e:
            print(f"   (Download failed: {e})")
    else:
         # Simulate checking local library
         if not os.path.exists(target_path):
             os.makedirs(target_path, exist_ok=True)
         print(f"   ⬇️ Acquired local skill: {target_path}")

    # C. EVALUATE
    print(f"   ⚖️ Evaluating Skill Quality...")
    try:
        report = client.evaluate(target=target_path) 
        
        # Output format based on user request: All 5 dimensions
        # {'safety': {'level': '...'}, 'completeness': {'level': '...'}, ...}
        
        dims = ['safety', 'completeness', 'executability', 'modifiability', 'cost_awareness']
        results_str = []
        for dim in dims:
            level = report.get(dim, {}).get('level', 'N/A')
            results_str.append(f"{dim.capitalize()}: {level}")
        
        print(f"      {' | '.join(results_str)}")
        
    except Exception as e:
        # Add logic to determine error cause
        error_msg = str(e)
        if "API_KEY" in error_msg or "api_key" in error_msg:
            print(f"      ⚠️ Evaluation skipped: Missing OpenAI API_KEY. Please set os.environ['API_KEY'].")
        else:
            print(f"      ⚠️ Evaluation failed: {error_msg}")

    active_skills[task['step']] = skill_name


# D. ANALYZE
print(f"\n🔗 **Analyzing Skill Relationships** (SkillNet Analyzer)...")
try:
    relationships = client.analyze(skills_dir=SKILLS_LIB, save_to_file=False)
    
    if relationships and isinstance(relationships, list):
        print(f"   Found {len(relationships)} dependencies between skills.")
        for r in relationships[:3]:
            # Parsing: {"source": "A", "target": "B", "type": "depend_on", "reason": "..."}
            src = r.get('source', 'Unknown')
            rtype = r.get('type', 'related_to')
            tgt = r.get('target', 'Unknown')
            print(f"   - {src} --[{rtype}]--> {tgt}")
    else:
        print("   No complex dependencies detected.")
        
except Exception as e:
    # Add logic to determine error cause
    error_msg = str(e)
    if "API_KEY" in error_msg or "api_key" in error_msg:
         print(f"   ⚠️ Analysis skipped: Missing OpenAI API_KEY. Relationship inference requires LLM.")
    else:
         print(f"   ⚠️ Analysis failed: {error_msg}")

print(f"\n✨ **Orchestration Complete**: {len(active_skills)} skills ready for execution.")


🤖 **AI Scientist Agent Initialized**

📝 **User Mission**: "Analyze single-cell RNA-seq data to identify potential cancer therapeutic targets, validate them against clinical databases, and write a summary report."

🚀 **SkillNet Orchestration Started**

--- [Phase 1: Data Processing] ---
🔍 Searching SkillNet for: 'cellxgene'...
   ✅ Found Skill: cellxgene-census (Stars: 15976)
   ⬇️ Downloading to: ./active_skills_library\cellxgene-census...
   ⚖️ Evaluating Skill Quality...
      Safety: Good | Completeness: Good | Executability: Good | Modifiability: Good | Cost_awareness: Good

--- [Phase 2: Mechanism Analysis] ---
🔍 Searching SkillNet for: 'kegg'...
   ✅ Found Skill: kegg-database (Stars: 17598)
   ⬇️ Downloading to: ./active_skills_library\kegg-database...
   ⚖️ Evaluating Skill Quality...
      Safety: Good | Completeness: Good | Executability: Average | Modifiability: Good | Cost_awareness: Good

--- [Phase 3: Target Validation] ---
🔍 Searching SkillNet for: 'gget'...
   ✅ Found S

## Execution Phase 1: Data Processing
**Active Skill**: `cellxgene-census`

The Agent uses the `cellxgene-census` skill to query and retrieve single-cell data.
*(Note: For this demo, we simulate the retrieval of a dataset containing specific marker genes)*


In [36]:
current_skill = active_skills[1] 
print(f"▶️ **Running Skill**: [{current_skill}]...")

# --- Skill Implementation: cellxgene-census ---
# In a fully autonomous run, the Agent would execute:
# > cellxgene_census.get_anndata(organism="Homo sapiens", measurement_name="RNA")

# Simulating Data Retrieval & Processing
adata = sc.datasets.blobs(n_variables=1000, n_observations=300, n_centers=3, cluster_std=1.0, random_state=42)
adata.X = np.abs(adata.X) # Ensure non-negative

# 1. Quality Control (Mock QC)
# FIX: Define 'mt' (mitochondrial) genes for QC metrics to avoid KeyError
adata.var['mt'] = np.random.choice([True, False], size=adata.n_vars, p=[0.1, 0.9])

# Standard Scanpy Pipeline (Process)
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=20)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# Identify Marker
try:
    top_marker_synthetic = sc.get.rank_genes_groups_df(adata, group='0').iloc[0]['names']
except KeyError:
    top_marker_synthetic = sc.get.rank_genes_groups_df(adata, group=adata.obs['leiden'].cat.categories[0]).iloc[0]['names']

target_gene_symbol = "EGFR" # Simulating finding EGFR in the census data
# --- End Skill Implementation ---

print(f"✅ **[{current_skill}] Completed**. Retrieved and processed dataset. Identified Target: '{target_gene_symbol}'")


▶️ **Running Skill**: [cellxgene-census]...
normalizing counts per cell
    finished ({time_passed})
computing PCA
    with n_comps=50
    finished (0:00:00)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:00)
running Leiden clustering
    finished: found 3 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)
ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to b

## Execution Phase 2: Mechanism Analysis
**Active Skill**: `kegg-database`

The Agent uses `scripts/kegg_api.py` from the `kegg-database` skill to map the gene to biological pathways.


In [37]:
current_skill = active_skills[2]
print(f"▶️ **Running Skill**: [{current_skill}] (Input: {target_gene_symbol})...")

# --- Skill Implementation: kegg-database ---
# Emulating logic from `active_skills_library/kegg-database/scripts/kegg_api.py`
def run_kegg_lookup(gene):
    try:
        # 1. Look up Gene ID
        find_url = f"http://rest.kegg.jp/find/genes/{gene}"
        resp = requests.get(find_url).text
        kegg_id = resp.split('\t')[0] if "hsa:" in resp else None
        
        if not kegg_id: return []
        
        # 2. Retrieve Pathways
        link_url = f"http://rest.kegg.jp/link/pathway/{kegg_id}"
        pathways = [line.split('\t')[1] for line in requests.get(link_url).text.split('\n') if line]
        return pathways
    except:
        return []

associated_pathways = run_kegg_lookup(target_gene_symbol)
# --- End Skill Implementation ---

print(f"✅ **[{current_skill}] Completed**. Mapped {target_gene_symbol} to {len(associated_pathways)} pathways.")


▶️ **Running Skill**: [kegg-database] (Input: EGFR)...
✅ **[kegg-database] Completed**. Mapped EGFR to 50 pathways.


## Execution Phase 3: Target Validation
**Active Skill**: `gget`

The Agent uses `gget` (Gene enhancement tool) to validate the target's associations in the Open Targets Platform.


In [38]:
current_skill = active_skills[3]
print(f"▶️ **Running Skill**: [{current_skill}]...")

# --- Skill Implementation: gget (Open Targets Module) ---
# Emulating `gget.opentargets` behavior
def run_gget_opentargets(gene_symbol):
    # gget typically wraps the GraphQL API for Open Targets
    url = "https://api.platform.opentargets.org/api/v4/graphql"
    
    # 1. ID Mapping
    search_q = """query($q:String!){search(queryString:$q,entityNames:["target"],page:{index:0,size:1}){hits{id}}}"""
    r1 = requests.post(url, json={"query": search_q, "variables": {"q": gene_symbol}})
    
    try:
        ensembl_id = r1.json()['data']['search']['hits'][0]['id']
    except:
        return None

    # 2. Association Query
    details_q = """query($id:String!){target(ensemblId:$id){approvedSymbol approvedName associatedDiseases(page:{index:0,size:5}){rows{disease{name} score}}}}"""
    r2 = requests.post(url, json={"query": details_q, "variables": {"id": ensembl_id}})
    
    return r2.json().get('data', {}).get('target')

ot_data = run_gget_opentargets(target_gene_symbol)
# --- End Skill Implementation ---

if ot_data:
    top_d = ot_data['associatedDiseases']['rows'][0]['disease']['name']
    print(f"✅ **[{current_skill}] Completed**. Validated target. Top association: {top_d}")
else:
    print(f"⚠️ **[{current_skill}] Completed**. No data found.")


▶️ **Running Skill**: [gget]...
✅ **[gget] Completed**. Validated target. Top association: non-small cell lung carcinoma


## Execution Phase 4: Reporting
**Active Skill**: `citation-management`

The Agent generates the final report using the collected data, utilizing the `citation-management` skill to ensure proper referencing.


In [39]:
current_skill = active_skills[4]
print(f"▶️ **Running Skill**: [{current_skill}]...")

# --- Skill Implementation: citation-management ---
# Simulating `scripts/format_bibtex.py` and `scripts/validate_citations.py` from the skill
def format_citation(source, year, title):
    return f"[{source}, {year}]"

def generate_report_with_citations(gene, pathways, ot_info):
    # Mock Citations
    ref_scanpy = format_citation("Wolf et al.", 2018, "Scanpy")
    ref_kegg = format_citation("Kanehisa", 2000, "KEGG")
    ref_ot = format_citation("Open Targets", 2024, "Platform")
    
    disease_list = [d['disease']['name'] for d in ot_info['associatedDiseases']['rows'][:3]] if ot_info else []
    
    report_md = f"""
# Scientific Discovery Report: {gene} Analysis
*Generated by SkillNet AI Scientist*

## 1. Data Processing
**Skill**: `{active_skills[1]}`
Using standardized single-cell analysis pipelines {ref_scanpy}, we identified **{gene}** as a significant marker.

## 2. Biological Mechanism
**Skill**: `{active_skills[2]}`
Pathway enrichment analysis {ref_kegg} identified {len(pathways)} associated pathways, linking {gene} to key biological processes:
{chr(10).join([f'- {p}' for p in pathways[:3]])}

## 3. Therapeutic Validation
**Skill**: `{active_skills[3]}`
Cross-referencing with clinical databases {ref_ot} confirms therapeutic relevance for:
{chr(10).join([f'- {d}' for d in disease_list])}

## 4. References (Managed by {active_skills[4]})
1. Wolf, F. A., et al. (2018). Scanpy: large-scale single-cell gene expression data analysis. Genome biology.
2. Kanehisa, M. & Goto, S. (2000). KEGG: kyoto encyclopedia of genes and genomes. Nucleic acids research.
3. Open Targets Platform (2024). version 24.03.
    """
    return report_md

report_content = generate_report_with_citations(target_gene_symbol, associated_pathways, ot_data)
# --- End Skill Implementation ---

print(f"✅ **[{current_skill}] Completed**. Report with citations generated.")
display(Markdown(report_content))


▶️ **Running Skill**: [citation-management]...
✅ **[citation-management] Completed**. Report with citations generated.



# Scientific Discovery Report: EGFR Analysis
*Generated by SkillNet AI Scientist*

## 1. Data Processing
**Skill**: `cellxgene-census`
Using standardized single-cell analysis pipelines [Wolf et al., 2018], we identified **EGFR** as a significant marker.

## 2. Biological Mechanism
**Skill**: `kegg-database`
Pathway enrichment analysis [Kanehisa, 2000] identified 50 associated pathways, linking EGFR to key biological processes:
- path:hsa01521
- path:hsa01522
- path:hsa03272

## 3. Therapeutic Validation
**Skill**: `gget`
Cross-referencing with clinical databases [Open Targets, 2024] confirms therapeutic relevance for:
- non-small cell lung carcinoma
- lung adenocarcinoma
- cancer

## 4. References (Managed by citation-management)
1. Wolf, F. A., et al. (2018). Scanpy: large-scale single-cell gene expression data analysis. Genome biology.
2. Kanehisa, M. & Goto, S. (2000). KEGG: kyoto encyclopedia of genes and genomes. Nucleic acids research.
3. Open Targets Platform (2024). version 24.03.
    